# 02 - LightGBM forecast + SHAP drivers

**Goal:** see whether a learned model beats the baselines from notebook 01 on your data, and *why* it predicts
what it predicts. Expect modest results: you have ~110 rows. Honest "no better than mean7" is a valid finding.

**Concepts**
- *Gradient boosting*: many small trees, each correcting the previous ones' errors. Powerful, and with 110 rows
  it will memorise noise unless you constrain it hard (small `num_leaves`, large `min_child_samples`, few
  trees, learning rate low). LightGBM also handles NaN natively, so you do not need to fill gaps.
- *Time-series CV*: `sklearn.model_selection.TimeSeriesSplit` (training always precedes validation).
- *Feature leakage*: `sleep_score` and `bb_max` are *derived* from sleep and HRV. Using same-night
  `sleep_score` to predict same-night sleep is cheating. Use only what is known by the end of day `t`.
- *SHAP*: assigns each feature a signed contribution to each prediction. Great for "why was tomorrow predicted
  low?", but with ~100 rows the attributions are noisy: treat them as hypotheses, not facts.

**Rule:** features are defined in `api/app/ml/features.py`, NOT in this notebook. The nightly job must compute
exactly the same columns. Add a function there (lags, rolling stats, weekday, missing flags) and call it from `build`.

In [ ]:
# --- Setup (provided, working). Do not edit; fix things in api/app/ml/features.py instead. ---
import pathlib, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "api").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "api"))

from app.ml import features, evaluate          # the SAME modules the nightly job imports

raw = features.frame_from_csv(ROOT / "ml/data/daily_metrics.csv")   # one row per calendar day
feat = features.build(raw)                                          # baselines, z-scores, missing flags
print(f"{len(raw)} days, {raw.index.min().date()} -> {raw.index.max().date()}, "
      f"{int(raw['row_present'].sum())} with a row, {int(raw['sleep_hours'].isna().sum())} with no sleep value")
raw.tail()

## 1. Choose features (in `features.py`)
Brainstorm first. For each candidate say what it is, why it could matter, and how it could leak.
Candidates to consider: lagged sleep/HRV (1, 2, 3 days), 7-day rolling mean, deviation from baseline (`*_z`),
weekday, missing-flag, `n_metrics_present`. Manual inputs (alcohol, caffeine, RPE) arrive in Phase 4.

In [ ]:
# TODO: write the list of features and one sentence each (as comments). Then add them to features.py and re-run setup.

## 2. Build the dataset
Align features at day `t` with the target from notebook 01 (`make_target`). Drop only rows with no target; keep NaN in features.

In [ ]:
# TODO: X = feat[...your columns...]; y = target; keep the dates so you can plot later.

## 3. Walk-forward cross-validation
Use `TimeSeriesSplit` (try `n_splits=5`). For each fold: fit on the past, predict the future, store `evaluate.mae`.
Compare mean fold MAE to naive/mean7 **on the same dates**.

Starter parameters to experiment with (do not trust them; search around them):
`n_estimators` 100-300, `learning_rate` 0.03-0.05, `num_leaves` 4-8, `min_child_samples` 8-15,
`subsample`/`colsample_bytree` ~0.8, `reg_lambda` 1-5.

In [ ]:
# TODO: from lightgbm import LGBMRegressor; from sklearn.model_selection import TimeSeriesSplit

## 4. Does it beat the baselines?
Use the last-30-days window and `evaluate.skill`. Also try a *sanity check*: shuffle the target within each training fold. If the model still scores well, you have leakage.

In [ ]:
# TODO: table model | MAE | n | skill; plus the shuffled-target check.

## 5. SHAP drivers
1. Fit the final model on all data up to yesterday.
2. `shap.TreeExplainer(model)`; `explainer.shap_values(X)`.
3. Global: bar/summary plot, which features matter on average?
4. Local: explain **one** day (tomorrow's prediction). Which 2-3 features pushed it up or down, and by how much (in hours)?
5. Stability: refit on a different 80% of the days. Do the top features stay the same? If not, say so.

In [ ]:
# TODO: import shap; ...

## 6. Save the model artifact (contract in `ml/README.md`)
When you have a candidate, save it as `ml/models/sleep_hours_h1_YYYY-MM-DD.joblib` with a dict containing
`model, feature_names, target, horizon, trained_through, kind, cv_mae, features_version`.

In [ ]:
# TODO: joblib.dump({...}, ROOT / 'ml/models' / f'sleep_hours_h1_{...}.joblib')

## Results (fill in)
| Model | MAE (h) | n | Skill vs naive |
|---|---|---|---|
| naive | | | 0 |
| mean7 | | | |
| LightGBM | | | |

**Top drivers (with a caveat about sample size):**